# Analiza dostępności przestrzennej i potencjału ekspansji sieci Żabka w Polsce

**Autorzy:**
- Jakub Rosiak 251620
- Mateusz Kosowski 251558
- Nikodem Nowak 251598

### O projekcie
Projekt zajmuje się analizą rozmieszczenia sieci sklepów Żabka w Polsce. Wykorzystując zbiór danych o lokalizacji blisko 10 tysięcy placówek (stan na 2024 r.) oraz oficjalne dane demograficzne z Narodowego Spisu Powszechnego 2021 (siatka kilometrowa GUS), przeprowadziliśmy analizę dostępności usług.

Kluczowym elementem projektu jest integracja danych punktowych z danymi rastrowymi (populacja) przy użyciu algorytmów przestrzennych (m.in. *Spatial Join*, *KDTree*). Pozwoliło to nie tylko na ocenę obecnego stanu nasycenia rynku, ale przede wszystkim na wyznaczenie precyzyjnych rekomendacji dla nowych otwarć.

### Cele i zakres analizy

**1. Analiza Statystyczna**
*   Identyfikacja struktury usług dodatkowych (np. Żabka Café, usługi pocztowe).
*   Ranking nasycenia placówkami w podziale na województwa i największe ośrodki miejskie.
*   Badanie korelacji: gęstość zaludnienia vs liczba sklepów.

**2. Zaawansowana Analityka Przestrzenna**
*   **Obliczenia dostępności:** Wyznaczenie dystansu do najbliższej placówki dla każdej zamieszkanej komórki siatki w Polsce.
*   **Krzywa pokrycia:** Oszacowanie, jaki procent populacji kraju ma dostęp do sklepu w promieniu 1 km.
*   **Obciążenie sieci:** Analiza wskaźnika w celu wykrycia obszarów potencjalnie przeciążonych.

**3. Business Intelligence i Wizualizacja**
*   **Wykrywanie "białych plam":** Algorytmiczne wyznaczenie obszarów *underserved* (wysoka populacja, brak sklepu w pobliżu) i nadanie im priorytetów inwestycyjnych.
*   **Dashboard mapowy:** Stworzenie interaktywnej mapy wielowarstwowej (Folium) umożliwiającej dynamiczną eksplorację wyników (heatmapy, klastry, strefy zasięgu).


In [7]:
# Importy
import os
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import MarkerCluster, HeatMap
from scipy.spatial import cKDTree
from scipy import stats
from branca.element import MacroElement
from jinja2 import Template


# Konfiguracja wyglądu Seaborn i Matplotlib
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#333',
    'grid.color': '#ddd',
    'text.color': '#222',
    'figure.figsize': (14, 8)
})

# Tworzenie katalogu na output
os.makedirs("output", exist_ok=True)

# Ignorujemy ostrzeżenia aby konsola była czysta
warnings.filterwarnings('ignore')

W naszym projekcie posługujemy się dwoma różnymi systemami współrzędnych przestrzennych (CRS) definiowanych przez standard EPSG.

<b>EPSG:4326</b> to globalny system układu współrzędnych geograficznych, w którym jednostką są stopnie (długość i szerokość geograficzna). W naszym projekcie służy on do:

   1. Wizualizacji interaktywnej: Jest to standardowy układ obsługiwany przez bibliotekę Folium
   2. Przechowywania surowych danych: Współrzędne sklepów w pliku wejściowym zapisane są jako szerokość (lat) i długość (lng).

Natomiast <b> EPSG:2180</b> to system prostokątnych współrzędnych płaskich zaprojektowany specjalnie dla obszaru Polski. W naszym projekcie służy on do:
   1. Precyzyjnych obliczeń odległości: Ponieważ jednostką w tym układzie jest metr, pozwala on na dokładne wyliczenie dystansu między mieszkańcami a najbliższym sklepem Żabka
   2. Złączeń przestrzennych (Spatial Join): Umożliwia poprawne przypisanie sklepów do komórek siatki populacyjnej GUS, która natywnie korzysta z tego formatu.
   3. Analizy zagęszczenia: Jest niezbędny do rzetelnego operowania na danych o populacji

In [8]:
# Wczytywanie danych
df_shops = pd.read_csv("data/zabka_shops.csv")
df_shops = df_shops[
    (df_shops['lat'].between(49, 55)) &
    (df_shops['lng'].between(14, 25))
].copy()
df_shops['services'] = df_shops['services'].fillna('').astype(str)

# Przygotowanie GeoDataFrame w układzie 4326 dla Folium
gdf_shops_4326 = gpd.GeoDataFrame(
    df_shops,
    geometry=gpd.points_from_xy(df_shops.lng, df_shops.lat),
    crs="EPSG:4326"
)

# # Siatka GUS z populacją
try:
    gdf_population = gpd.read_file("data/GRID_NSP2021_RES/GRID_NSP2021_RES.shp")
    gdf_population.set_crs(epsg=2180, allow_override=True, inplace=True)
    gdf_shops_2180 = gdf_shops_4326.to_crs(epsg=2180)
    print(f"✓ Siatka GUS: {len(gdf_population)} komórek")
except Exception as e:
    print(f"✗ Błąd wczytywania siatki GUS: {e}")
    raise SystemExit(1)


✓ Siatka GUS: 315857 komórek


### Obliczenia przestrzenne i analiza potencjału
W tej sekcji przeprowadzamy kluczowe operacje na danych geoprzestrzennych, które posłużą do budowy wizualizacji i wyciągnięcia wniosków biznesowych:

1.  **Złączenie przestrzenne (Spatial Join):** Przypisujemy każdy sklep do odpowiedniej komórki siatki kilometrowej GUS, aby obliczyć zagęszczenie placówek (liczba sklepów w danej strefie).
2.  **Analiza odległości (KDTree):** Dla środka każdej komórki siatki z populacją obliczamy dystans w linii prostej do najbliższej Żabki. Wykorzystujemy algorytm *cKDTree* dla zapewnienia wysokiej wydajności obliczeń.
3.  **Kategoryzacja dostępności:** Dzielimy obszary na strefy dostępności, co ułatwi późniejszą wizualizację.
4.  **Metryki biznesowe:**
    *   Wyznaczamy wskaźnik obciążenia sklepu (*people per shop*).
    *   Identyfikujemy obszary "niedobsłużone" (*underserved*) – komórki o wysokiej populacji (>500 osób), ale oddalone od sklepu o ponad 1.5 km, nadając im priorytet inwestycyjny.


In [9]:
# Sklepy w komórkach siatki
joined = gpd.sjoin(gdf_shops_2180, gdf_population, how="inner", predicate="within")
shop_counts = joined['index_right'].value_counts()
gdf_population['shop_count'] = 0
gdf_population.loc[shop_counts.index, 'shop_count'] = shop_counts

# Dystans do najbliższej Żabki (KDTree)
shop_coords = np.column_stack([
    gdf_shops_2180.geometry.x,
    gdf_shops_2180.geometry.y
])
tree = cKDTree(shop_coords)
centroids = gdf_population.geometry.centroid
grid_coords = np.column_stack([centroids.x, centroids.y])
distances, _ = tree.query(grid_coords)
gdf_population['distance_m'] = distances
gdf_population['distance_km'] = distances / 1000

# Kategorie dystansu
bins = [0, 1000, 2000, 5000, np.inf]
labels = ['< 1km', '1-2km', '2-5km', '> 5km']
gdf_population['distance_cat'] = pd.cut(gdf_population['distance_m'], bins=bins, labels=labels)

# Ludzie na sklep (tylko tam gdzie są sklepy) - do choropleth
mask_shops = gdf_population['shop_count'] > 0
gdf_population['people_per_shop'] = np.nan
gdf_population.loc[mask_shops, 'people_per_shop'] = (
    gdf_population.loc[mask_shops, 'RES'] / gdf_population.loc[mask_shops, 'shop_count']
)

# Obszary niedostępne - duży potencjał inwestycyjny (>= 500 osób i > 1.5 km)
underserved = gdf_population[
    (gdf_population['RES'] >= 500) &
    (gdf_population['distance_m'] > 1500)
].copy()
underserved['priority'] = underserved['RES'] * underserved['distance_km']
underserved = underserved.sort_values('priority', ascending=False)

### Statystyki dostępności i pokrycia populacji
W tej sekcji agregujemy wyniki analizy przestrzennej, aby uzyskać ogólny obraz dostępności sieci dla populacji Polski. Obliczamy kluczowe wskaźniki:

*   **Całkowita populacja:** Suma ludności ze wszystkich komórek siatki (punkt odniesienia).
*   **Populacja w strefach zasięgu:** Liczba osób mieszkających w promieniu 1 km (spacer), 2 km (krótki dojazd) oraz 5 km od najbliższego sklepu. Pozwala to ocenić penetrację rynku.
*   **Populacja wykluczona (Far Pop):** Liczba osób mieszkających powyżej 5 km od najbliższej placówki – potencjalne "białe plamy" lub obszary wiejskie o niskim priorytecie.
*   **Średni i medianowy dystans:** Globalne metryki pokazujące, jak daleko statystyczny Polak ma do najbliższej Żabki.


In [10]:
# Statystyki dostępności sklepów
total_pop = gdf_population['RES'].sum()
pop_within_1km = gdf_population[gdf_population['distance_m'] <= 1000]['RES'].sum()
pop_within_2km = gdf_population[gdf_population['distance_m'] <= 2000]['RES'].sum()
pop_within_5km = gdf_population[gdf_population['distance_m'] <= 5000]['RES'].sum()
far_pop = gdf_population[gdf_population['distance_m'] > 5000]['RES'].sum()
avg_distance = gdf_population['distance_m'].mean()
median_distance = gdf_population['distance_m'].median()

### Przygotowanie danych do wizualizacji
W tym kroku przetwarzamy surowe dane na formaty wymagane przez biblioteki *Matplotlib* i *Seaborn* w celu wygenerowania czytelnych wykresów:

1.  **Analiza usług:** Rozbijamy ciągi znaków z kodami usług (np. 'ZBC', 'LOT'), mapujemy je na pełne nazwy i tworzymy ranking najczęściej oferowanych udogodnień.
2.  **Agregacja geograficzna:** Wyznaczamy liczbę placówek w podziale na województwa oraz Top 10 miast.
3.  **Agregacja demograficzna:** Sumujemy populację w ramach wyznaczonych wcześniej kategorii odległości (np. ile osób mieszka < 1km od sklepu).
4.  **Konfiguracja wizualna:** Definiujemy spójną paletę kolorów dla kategorii dystansu oraz funkcję pomocniczą `add_bar_labels` do nanoszenia wartości liczbowych na słupki wykresów.


In [11]:
# Przygotowanie danych do wykresów
services_exploded = df_shops['services'].str.split(',').explode().str.strip()
services_counts = services_exploded.value_counts()

legend_map = {
    'ZBC': 'Żabka Café', 'ODP': 'Odpiek Pieczywa', 'PAC': 'Paczki',
    'TER': 'Płatność Kartą', 'GSM': 'Doładowania', 'KPO': 'Karty Podarunkowe',
    'RAC': 'Rachunki', 'REJ': 'Rejestracja SIM', 'DEN': 'Usługi Energetyczne',
    'LOT': 'Lotto', 'BIH': 'Cashback', 'DKM': 'Karta Miejska'
}
services_names = [legend_map.get(code, code) for code in services_counts.index]

top_cities = df_shops['city'].value_counts().head(10)
voivodeships = df_shops['voivodeship'].value_counts()
pop_by_dist = gdf_population.groupby('distance_cat')['RES'].sum().reindex(labels, fill_value=0)

# Kolory dla dystansów
color_map = {'< 1km': '#2ecc71', '1-2km': '#f1c40f', '2-5km': '#e67e22', '> 5km': '#e74c3c'}
dist_colors = [color_map[label] for label in labels]

# Funkcja do dodawania etykiet na słupkach
def add_bar_labels(ax, fontsize=9, offset=0):
    for p in ax.patches:
        w = p.get_width()
        if w > 0:
            ax.text(w + offset, p.get_y() + p.get_height()/2, f'{int(w)}',
                    ha='left', va='center', fontsize=fontsize)

### Wizualizacja wyników
W tej sekcji generujemy zestaw statycznych wykresów podsumowujących, które zostaną zapisane do plików PNG. Wizualizacje podzielone są na dwa panele:

**1. Panel kompletny (`analiza_kompletna.png`):**
*   **Usługi i lokalizacje:** Ranking najpopularniejszych usług dodatkowych oraz rozkład sklepów w województwach i największych miastach.
*   **Dostępność:** Wykres słupkowy populacji w strefach odległości oraz krzywa skumulowana pokazująca, jaki procent Polaków ma Żabkę w promieniu X km.
*   **Potencjał:** Scatter plot (punktowy) identyfikujący komórki o dużej populacji, ale słabej dostępności (kandydaci na nowe sklepy).

**2. Panel rozszerzony (`analiza_rozszerzona.png`):**
*   **Histogram dystansów:** Szczegółowy rozkład odległości ważony populacją.
*   **Korelacja:** Badanie zależności między liczbą mieszkańców w komórce siatki a liczbą znajdujących się w niej sklepów.


In [12]:
fig = plt.figure(figsize=(18, 12), facecolor='white')
gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)

# 1. Usługi dodatkowe
ax1 = fig.add_subplot(gs[0, 0])
sns.barplot(x=services_counts.values, y=services_names, palette="viridis", ax=ax1)
ax1.set_title("Usługi dodatkowe w sklepach", fontweight='bold', fontsize=13)
ax1.set_xlabel("Liczba placówek")
add_bar_labels(ax1)

# 2. Województwa
ax2 = fig.add_subplot(gs[0, 1])
sns.barplot(x=voivodeships.values, y=voivodeships.index, palette="mako", ax=ax2)
ax2.set_title("Liczba sklepów wg województw", fontweight='bold', fontsize=13)
ax2.set_xlabel("Liczba sklepów")
add_bar_labels(ax2)

# 3. Top miasta
ax3 = fig.add_subplot(gs[1, 0])
sns.barplot(x=top_cities.values, y=top_cities.index, palette="rocket", ax=ax3)
ax3.set_title("Top 10 miast", fontweight='bold', fontsize=13)
ax3.set_xlabel("Liczba sklepów")
add_bar_labels(ax3)

# 4. Populacja wg dystansu
ax4 = fig.add_subplot(gs[1, 1])
pop_by_dist.plot(kind='bar', ax=ax4, color=dist_colors)
ax4.set_title("Populacja wg dystansu do Żabki", fontweight='bold', fontsize=13)
ax4.set_xlabel("Dystans")
ax4.set_ylabel("Liczba mieszkańców")
ax4.set_xticklabels(labels, rotation=0)
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x/1e6)}M'))

# 5. Krzywa pokrycia
ax5 = fig.add_subplot(gs[2, 0])
pop_data = gdf_population[gdf_population['RES'] > 0].copy()
pop_data = pop_data.sort_values('distance_km')
pop_data['cum_pop'] = pop_data['RES'].cumsum()
pop_data['cum_share'] = pop_data['cum_pop'] / pop_data['RES'].sum()
ax5.plot(pop_data['distance_km'], pop_data['cum_share'], color='#4c78a8', linewidth=2.5)
for d in [1, 2, 5]:
    ax5.axvline(d, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax5.set_xlim(0, 10)
ax5.set_ylim(0, 1)
ax5.set_title("Krzywa pokrycia populacji", fontweight='bold', fontsize=13)
ax5.set_xlabel("Dystans do Żabki (km)")
ax5.set_ylabel("Udział populacji")
ax5.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x*100)}%'))

# 6. Scatter - ludność vs dystans
ax6 = fig.add_subplot(gs[2, 1])
sample = gdf_population[gdf_population['RES'] > 0].sample(
    min(5000, len(gdf_population)), random_state=42
)
ax6.scatter(sample['distance_km'], sample['RES'], alpha=0.3, s=8, color='#72b7b2')
ax6.axvline(1.5, color='#e45756', linestyle='--', linewidth=1.5, label='Próg 1.5 km')
ax6.axhline(500, color='#f58518', linestyle='--', linewidth=1.5, label='Próg 500 osób')
ax6.set_xlim(0, 10)
ax6.set_title("Ludność vs dystans (próbka 5000)", fontweight='bold', fontsize=13)
ax6.set_xlabel("Dystans (km)")
ax6.set_ylabel("Ludność w komórce")
ax6.legend(frameon=True, loc='upper right')

plt.savefig('output/analiza_kompletna.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close()




fig2 = plt.figure(figsize=(18, 6), facecolor='white')
gs2 = fig2.add_gridspec(1, 2, wspace=0.2)

# 7. HISTOGRAM DYSTANSÓW
ax7 = fig2.add_subplot(gs2[0, 0])
pop_with_res = gdf_population[gdf_population['RES'] > 0]
ax7.hist(pop_with_res['distance_km'], bins=50, weights=pop_with_res['RES'],
         color='#4c78a8', edgecolor='white', alpha=0.8)
ax7.axvline(1, color='#2ecc71', linestyle='--', linewidth=2, label='1 km')
ax7.axvline(2, color='#f1c40f', linestyle='--', linewidth=2, label='2 km')
ax7.axvline(5, color='#e74c3c', linestyle='--', linewidth=2, label='5 km')
ax7.set_title("Rozkład dystansów do najbliższej Żabki", fontweight='bold', fontsize=13)
ax7.set_xlabel("Dystans (km)")
ax7.set_ylabel("Liczba mieszkańców")
ax7.set_xlim(0, 15)
ax7.legend(loc='upper right', frameon=True)
ax7.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x/1e6)}M'))

# 8. KORELACJA: GĘSTOŚĆ VS SKLEPY
ax8 = fig2.add_subplot(gs2[0, 1])
# Agregacja per komórka - ile mieszkańców na km2 vs ile sklepów w promieniu 1km
grid_with_pop = gdf_population[gdf_population['RES'] > 100].copy()
sample_corr = grid_with_pop.sample(min(3000, len(grid_with_pop)), random_state=42)

ax8.scatter(sample_corr['RES'], sample_corr['shop_count'], alpha=0.4, s=15, color='#72b7b2')

# Linia trendu
mask_valid = (sample_corr['shop_count'] > 0) & (sample_corr['RES'] > 0)
if mask_valid.sum() > 10:
    slope, intercept, r_value, _, _ = stats.linregress(
        sample_corr.loc[mask_valid, 'RES'],
        sample_corr.loc[mask_valid, 'shop_count']
    )
    x_line = np.linspace(sample_corr['RES'].min(), sample_corr['RES'].max(), 100)
    y_line = slope * x_line + intercept
    ax8.plot(x_line, y_line, color='#e45756', linewidth=2, label=f'R² = {r_value**2:.2f}')
    ax8.legend(loc='upper right', frameon=True)

ax8.set_title("Korelacja: liczba mieszkańców vs sklepy", fontweight='bold', fontsize=13)
ax8.set_xlabel("Liczba mieszkańców w komórce")
ax8.set_ylabel("Liczba sklepów")

plt.tight_layout()
plt.savefig('output/analiza_rozszerzona.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close()

### Generowanie interaktywnej mapy analitycznej
Ostatnim etapem projektu jest stworzenie zaawansowanej, wielowarstwowej mapy w bibliotece *Folium*, która integruje wszystkie przeprowadzone analizy. Mapa umożliwia dynamiczne przełączanie widoków w panelu sterowania (prawy górny róg) i obejmuje następujące warstwy:

1.  **🐸 Sklepy Żabka (Klastry):** Wszystkie placówki zgrupowane w klastry, które rozdzielają się przy przybliżaniu mapy.
2.  **⚠️ Obszary niedostępne:** Wizualizacja "białych plam" – miejsc o wysokim zaludnieniu (>500 osób), ale oddalonych od najbliższego sklepu o ponad 1.5 km (kolorowe okręgi).
3.  **🔥 Heatmapa:** Mapa ciepła pokazująca regiony o najwyższym zagęszczeniu sklepów.
4.  **📊 Obciążenie (Choropleth):** Kartogram pokazujący wskaźnik *people per shop* w komórkach siatki, co pozwala zidentyfikować potencjalnie przeciążone placówki.
5.  **🎯 Strefy dostępności:** Wizualizacja ogólnego pokrycia kraju – każdy zamieszkany punkt siatki to "bąbelek" o kolorze zależnym od dystansu do sklepu.
6.  **🏆 Top 10 Lokalizacji:** Oznaczone gwiazdkami konkretne miejsca zarekomendowane do otwarcia nowych placówek na podstawie algorytmu priorytetyzacji.

Dodatkowo zaimplementowano niestandardową obsługę dynamicznej legendy (JavaScript) oraz stylizację CSS dla kontrolki warstw.


In [13]:
main_map = folium.Map(location=[52.0, 19.0], zoom_start=6, tiles=None)

for name, tiles, show in [
    ("Jasne", "CartoDB positron", True),
    ("OSM", "OpenStreetMap", False),
    ("Ciemne", "CartoDB dark_matter", False)
]:folium.TileLayer(tiles=tiles, name=f"Tło: {name}", control=True, show=show).add_to(main_map)

# WARSTWA 1: Sklepy Żabka (klastry)
shops_layer = folium.FeatureGroup(name="🐸 Sklepy Żabka (wszystkie)", show=True)
cluster = MarkerCluster().add_to(shops_layer)
for row in df_shops.itertuples():
    folium.Marker(
        [row.lat, row.lng],
        tooltip=f"<b>{row.city}</b><br>{row.address}",
        icon=folium.Icon(color="green", icon="frog", prefix="fa")
    ).add_to(cluster)
shops_layer.add_to(main_map)

# WARSTWA 2: Obszary niedostępne - pierwsze 250 aby mapa lepiej wyglądała
underserved_layer = folium.FeatureGroup(name="⚠️ Obszary niedostępne (>500 osób, >1.5km)", show=False)
underserved_4326 = underserved.head(250).to_crs(epsg=4326)

for _, row in underserved_4326.iterrows():
    c = row.geometry.centroid
    color = color_map.get(row.distance_cat, '#999')
    radius = min(18, max(6, row.RES / 200))

    folium.CircleMarker(
        [c.y, c.x],
        radius=radius,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        weight=1.5,
        popup=f"<b>Niedostępny obszar</b><br>Ludność: {int(row.RES)}<br>Dystans: {int(row.distance_m)}m ({row.distance_km:.1f} km)<br>Priorytet: {row.priority:.0f}",
        tooltip=f"💡 {int(row.RES)} osób | {int(row.distance_m)}m"
    ).add_to(underserved_layer)

underserved_layer.add_to(main_map)

# WARSTWA 3: Heatmapa gęstości
heat_data = [[row.lat, row.lng] for row in df_shops.itertuples()]
heatmap_layer = folium.FeatureGroup(name="🔥 Heatmapa gęstości sklepów", show=False)
HeatMap(
    heat_data,
    radius=15,
    blur=25,
    max_zoom=13,
    gradient={0.2: '#4c78a8', 0.5: '#72b7b2', 0.7: '#f58518', 0.9: '#e45756'}
).add_to(heatmap_layer)
heatmap_layer.add_to(main_map)

# WARSTWA 4: Choropleth obciążenie
gdf_with_shops = gdf_population[gdf_population['shop_count'] > 0].copy()
gdf_with_shops_4326 = gdf_with_shops.to_crs(epsg=4326)

max_val = gdf_with_shops['people_per_shop'].max()
bins_choro = [0, 1000, 2500, 5000, 7500, 10000, 12500, max_val + 1]
bins_choro = sorted(list(set([float(b) for b in bins_choro])))

choropleth = folium.Choropleth(
    geo_data=gdf_with_shops_4326,
    data=gdf_with_shops_4326,
    columns=['CODE', 'people_per_shop'],
    key_on='feature.properties.CODE',
    fill_color='YlOrRd',
    fill_opacity=0.6,
    line_opacity=0.1,
    legend_name='Mieszkańców na sklep',
    name="📊 Obciążenie sklepów (ludzie/sklep)",
    show=False,
    bins=bins_choro,
    nan_fill_color='white',
    nan_fill_opacity=0
)

tooltip_choropleth = folium.GeoJsonTooltip(
    fields=['RES', 'shop_count', 'people_per_shop'],
    aliases=['Ludność:', 'Sklepy:', 'Ludzi/sklep:'],
    localize=True
)
choropleth.geojson.add_child(tooltip_choropleth)
choropleth.add_to(main_map)

# Kontrolka legendy choropleth - widoczna tylko gdy warstwa jest włączona
class BindLegendToLayer(MacroElement):
    def __init__(self, layer):
        super().__init__()
        self.layer = layer
        self._template = Template("""
        {% macro script(this, kwargs) %}
            var map = {{this.layer._parent.get_name()}};
            var layer = {{this.layer.get_name()}};
            function getLegend() {
                return document.querySelector('.legend');
            }
            function setLegendVisibility() {
                var legend = getLegend();
                if (!legend) return;
                legend.style.display = map.hasLayer(layer) ? 'block' : 'none';
            }
            map.on('overlayadd', function(e) {
                if (e.layer === layer) setLegendVisibility();
            });
            map.on('overlayremove', function(e) {
                if (e.layer === layer) setLegendVisibility();
            });
            setTimeout(setLegendVisibility, 200);
        {% endmacro %}
        """)

main_map.add_child(BindLegendToLayer(choropleth))

# WARSTWA 5: Strefy dostępności (bąbelki)
zones_layer = folium.FeatureGroup(name="🎯 Strefy dostępności (wszystkie >200 osób)", show=False)
zones_all = gdf_population[gdf_population['RES'] > 200].to_crs(epsg=4326)

for _, row in zones_all.iterrows():
    c = row.geometry.centroid
    color = color_map.get(row.distance_cat, '#999')
    radius = min(8, max(2, row.RES / 400))

    folium.CircleMarker(
        [c.y, c.x],
        radius=radius,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.6,
        weight=0,
        tooltip=f"{row.distance_cat} | {int(row.RES)} osób"
    ).add_to(zones_layer)

zones_layer.add_to(main_map)

# WARSTWA 6: TOP 10 LOKALIZACJI NA NOWE SKLEPY
top_locations_layer = folium.FeatureGroup(name="🏆 Top 10 lokalizacji na nowy sklep", show=False)
top_10_locations = underserved.head(10).to_crs(epsg=4326)

for rank, (_, row) in enumerate(top_10_locations.iterrows(), 1):
    c = row.geometry.centroid

    folium.Marker(
        [c.y, c.x],
        popup=f"""
        <div style='width:200px'>
            <h4 style='color:#e74c3c; margin:5px 0'>🏆 #{rank} Priorytet ekspansji</h4>
            <b>Populacja:</b> {int(row.RES):,} osób<br>
            <b>Dystans do Żabki:</b> {row.distance_km:.1f} km<br>
            <b>Wskaźnik priorytetu:</b> {row.priority:,.0f}
        </div>
        """,
        tooltip=f"🏆 #{rank} | {int(row.RES):,} osób | {row.distance_km:.1f} km",
        icon=folium.Icon(color="red", icon="star", prefix="fa")
    ).add_to(top_locations_layer)

top_locations_layer.add_to(main_map)

In [14]:
# Stylizacja kontrolki warstw
css = """
<style>
.leaflet-control-layers {
    border-radius: 12px !important;
    box-shadow: 0 4px 16px rgba(0,0,0,0.25) !important;
    padding: 12px !important;
    background: rgba(255,255,255,0.97) !important;
    font-family: 'Segoe UI', sans-serif !important;
}
.leaflet-control-layers-toggle {
    width: 44px !important;
    height: 44px !important;
}
.leaflet-control-layers label {
    margin: 6px 0 !important;
    font-size: 13px !important;
}
</style>
"""
main_map.get_root().html.add_child(folium.Element(css))

folium.LayerControl(collapsed=False).add_to(main_map)
main_map.save('output/mapa_interaktywna.html')

### Podsumowanie i Wnioski

Zrealizowany projekt dostarczył kompleksowego obrazu sieci Żabka w Polsce, łącząc twarde dane lokalizacyjne z demografią (NSP 2021). Kluczowe rezultaty analizy to:

1.  **Ocena dostępności:**
    Dzięki analizie przestrzennej udało się ilościowo określić, jak duża część społeczeństwa ma dostęp do sklepu w zasięgu spaceru (<1 km). Analiza potwierdziła silną korelację między gęstością zaludnienia a liczbą placówek, wskazując na dojrzałość sieci w dużych ośrodkach miejskich.

2.  **Identyfikacja "Białych Plam":**
    Najważniejszą wartością biznesową projektu jest algorytm wykrywający obszary *underserved*. Zidentyfikowaliśmy konkretne komórki siatki kilometrowej, gdzie mieszka ponad **500 osób**, a dystans do najbliższej Żabki przekracza **1.5 km**. Są to obszary o wysokim priorytecie inwestycyjnym, które zostały wyróżnione na mapie.

3.  **Narzędzia analityczne:**
    Wygenerowana mapa interaktywna (`mapa_interaktywna.html`) stanowi funkcjonalne narzędzie wspierające decyzje. Warstwy takie jak *Heatmapa*, *Wskaźnik obciążenia (Choropleth)* czy rekomendowane *Top 10 Lokalizacji* pozwalają na szybką, wizualną weryfikację potencjału inwestycyjnego w dowolnym miejscu w Polsce.

Projekt pokazuje, że mimo dużej skali sieci (prawie 10 tys. punktów), wciąż istnieją na mapie Polski nisze rynkowe o wysokim potencjale nabywczym, możliwe do wykrycia dzięku integracji *Data Science* i *GIS*.